In [0]:
%sql

SHOW CATALOGS;

In [0]:
%sql

SHOW SCHEMAS IN data;

In [0]:
%sql
SHOW VOLUMES IN data.olympic_games;

In [0]:
%sql
LIST '/Volumes/data/olympic_games/raw_data';

In [0]:
volume_path = '/Volumes/data/olympic_games/raw_data'

files_in_volume = spark.sql(f"LIST '{volume_path}';")

files_in_volume.display()

In [0]:
DATA_PATH = "/Volumes/data/olympic_games/raw_data"

df_athletes = spark.read.csv(DATA_PATH + "/athlete_events.csv", header=True)

df_athletes.limit(2).display()

In [0]:
df_athletes.printSchema()

In [0]:
df_athletes = spark.read.csv(DATA_PATH + "/athlete_events.csv", header=True, inferSchema=True)

df_athletes.display()

## Define explicit schema

In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType, FloatType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", FloatType()),
  StructField("Weight", FloatType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_athletes_schema = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, schema=schema)
display(df_athletes_schema)

In [0]:
df_athletes_schema.groupBy("NOC", "Medal").count().filter("NOC IN ('SWE', 'FIN', 'NOR', 'DEN', 'ISL') AND Medal != 'NA'").sort("Medal","NOC").display()

In [0]:
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

fixar = spark.sql("""
        select
            sport, 
            count(medal) as Nr_medals
        from df_athletes_schema
        where noc IN 'SWE' AND medal IN ('Bronze', 'Silver', 'Gold')
        group by sport;
        """)

fixar.display()


In [0]:
%sql
SHOW SCHEMAS in data;

CREATE TABLE IF NOT EXISTS data.olympic_games.sweden_medals AS
(
    SELECT
        name,
        age,
        height,
        weight,
        year,
        sport,
        medal
    FROM
        df_athletes_schema
    WHERE
        noc = 'SWE'
        AND medal IN ('Bronze', 'Gold', 'Silver')
)
